In [11]:
import numpy as np
import pandas as pd
from scipy import stats

In [12]:
n_a, conv_a = 80_000, 1_600
n_b, conv_b = 80_000, 1_696

def build_user_level_ab(n_a, conv_a, n_b, conv_b, seed=42):
    rng = np.random.default_rng(seed)

    a_users = pd.DataFrame({
        'user_id': np.arange(1, n_a + 1),
        'group': 'A',
        'converted': np.r_[np.ones(conv_a, dtype=int), np.zeros(n_a - conv_a, dtype=int)]
    })

    b_users = pd.DataFrame({
        'user_id': np.arange(n_a + 1, n_a + n_b + 1),
        'group': 'B',
        'converted': np.r_[np.ones(conv_b, dtype=int), np.zeros(n_b - conv_b, dtype=int)]
    })

    df = pd.concat([a_users, b_users], ignore_index=True)
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)

df_ab = build_user_level_ab(n_a, conv_a, n_b, conv_b)
df_ab.head()

,user_id,group,converted
0,120477,B,0
1,32694,A,0
2,79959,A,0
3,76367,A,0
4,82344,B,0


In [13]:
def validate_ab_dataset(df, dataset_name='in_memory_ab_data'):
    """Базовая валидация для A/B данных перед расчетами."""
    required_cols = ['user_id', 'group', 'converted']

    print(f'Dataset: {dataset_name}')
    print(f'Shape: {df.shape}')

    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f'Missing required columns: {missing_cols}')

    missing = df[required_cols].isna().sum()
    print('Missing values in required cols:')
    print(missing.to_string())

    unique_target = set(df['converted'].dropna().unique().tolist())
    if not unique_target.issubset({0, 1}):
        raise ValueError('Column `converted` must contain only 0/1 values.')

    unique_groups = set(df['group'].dropna().unique().tolist())
    if not unique_groups.issubset({'A', 'B'}):
        raise ValueError('Column `group` must contain only A/B values.')

    dup_users = df['user_id'].duplicated().sum()
    print(f'Duplicate user_id count: {dup_users}')

    print('Validation passed.')

validate_ab_dataset(df_ab)

Dataset: in_memory_ab_data
Shape: (160000, 3)
Missing values in required cols:
user_id      0
group        0
converted    0
Duplicate user_id count: 0
Validation passed.


In [14]:
def srm_test_equal_split(df):
    """Проверка Sample Ratio Mismatch для ожидаемого сплита 50/50."""
    counts = df['group'].value_counts().reindex(['A', 'B'])
    observed = counts.values.astype(float)
    expected = np.array([observed.sum() / 2, observed.sum() / 2])

    chi2 = ((observed - expected) ** 2 / expected).sum()
    p_value = 1 - stats.chi2.cdf(chi2, df=1)

    return {
        'n_A': int(observed[0]),
        'n_B': int(observed[1]),
        'chi2': float(chi2),
        'p_value': float(p_value)
    }

srm = srm_test_equal_split(df_ab)
srm

{'n_A': 80000, 'n_B': 80000, 'chi2': 0.0, 'p_value': 1.0}

In [15]:
def ab_test_proportions(df, alpha=0.05):
    grouped = df.groupby('group')['converted'].agg(['sum', 'count']).reindex(['A', 'B'])

    x_a, n_a = grouped.loc['A', 'sum'], grouped.loc['A', 'count']
    x_b, n_b = grouped.loc['B', 'sum'], grouped.loc['B', 'count']

    p_a = x_a / n_a
    p_b = x_b / n_b

    # Uplift
    abs_uplift = p_b - p_a
    rel_uplift = abs_uplift / p_a if p_a > 0 else np.nan

    # Two-proportion z-test (pooled SE)
    p_pool = (x_a + x_b) / (n_a + n_b)
    se_pool = np.sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))
    z = (p_b - p_a) / se_pool
    p_value_two_sided = 2 * (1 - stats.norm.cdf(abs(z)))

    # CI for difference in proportions (unpooled SE)
    se_diff = np.sqrt((p_a * (1 - p_a) / n_a) + (p_b * (1 - p_b) / n_b))
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    ci_low = abs_uplift - z_alpha * se_diff
    ci_high = abs_uplift + z_alpha * se_diff

    return {
        'n_A': int(n_a),
        'x_A': int(x_a),
        'cr_A': float(p_a),
        'n_B': int(n_b),
        'x_B': int(x_b),
        'cr_B': float(p_b),
        'absolute_uplift': float(abs_uplift),
        'relative_uplift': float(rel_uplift),
        'z_score': float(z),
        'p_value_two_sided': float(p_value_two_sided),
        'ci_low': float(ci_low),
        'ci_high': float(ci_high)
    }

result = ab_test_proportions(df_ab, alpha=0.05)
result

{'n_A': 80000,
 'x_A': 1600,
 'cr_A': 0.02,
 'n_B': 80000,
 'x_B': 1696,
 'cr_B': 0.0212,
 'absolute_uplift': 0.0011999999999999997,
 'relative_uplift': 0.059999999999999984,
 'z_score': 1.6896532254015406,
 'p_value_two_sided': 0.09109431677976376,
 'ci_low': -0.0001919636068875312,
 'ci_high': 0.0025919636068875308}

In [16]:
summary = pd.DataFrame({
    'metric': [
        'CR A',
        'CR B',
        'Absolute uplift (pp)',
        'Relative uplift (%)',
        'z-score',
        'p-value (two-sided)',
        '95% CI low (pp)',
        '95% CI high (pp)',
        'SRM p-value'
    ],
    'value': [
        result['cr_A'] * 100,
        result['cr_B'] * 100,
        result['absolute_uplift'] * 100,
        result['relative_uplift'] * 100,
        result['z_score'],
        result['p_value_two_sided'],
        result['ci_low'] * 100,
        result['ci_high'] * 100,
        srm['p_value']
    ]
})

summary

,metric,value
0,CR A,2.000000
1,CR B,2.120000
2,Absolute uplift (pp),0.120000
3,Relative uplift (%),6.000000
4,z-score,1.689653
5,p-value (two-sided),0.091094
6,95% CI low (pp),-0.019196
7,95% CI high (pp),0.259196
8,SRM p-value,1.000000


## Интерпретация

Проверьте три условия перед решением:
- `SRM p-value > 0.05` (нет подозрительного перекоса трафика)
- `p-value < 0.05` (статистическая значимость)
- 95% CI для uplift не пересекает 0 и бизнес-эффект достаточный

Если условия выполнены, результат можно считать надежным для бизнес-решения.

In [17]:
def final_decision(srm_p, p_value, ci_low, ci_high, min_business_uplift_pp=0.2):
    checks = {
        'srm_ok': srm_p > 0.05,
        'stat_sig': p_value < 0.05,
        'ci_excludes_zero': (ci_low > 0) or (ci_high < 0),
        'business_effect_ok': (ci_low * 100) >= min_business_uplift_pp
    }

    launch = all(checks.values())
    return launch, checks

launch, checks = final_decision(
    srm_p=srm['p_value'],
    p_value=result['p_value_two_sided'],
    ci_low=result['ci_low'],
    ci_high=result['ci_high'],
    min_business_uplift_pp=0.2
)

print('Launch decision:', 'YES' if launch else 'NO')
print('Checks:', checks)

Launch decision: NO
Checks: {'srm_ok': True, 'stat_sig': False, 'ci_excludes_zero': False, 'business_effect_ok': False}


## Домашнее задание

1. Измените числа `n_a`, `conv_a`, `n_b`, `conv_b` и проверьте, когда тест перестает быть значимым.
2. Попробуйте сценарий с малым трафиком и тем же uplift.

#### Ссылки на A/B тестирование онлайн
- https://posthog.com/
- https://www.growthbook.io/
- https://abtestguide.com/calc/
